<a href="https://colab.research.google.com/github/Amelbnmbh/HR-Analysis-at-HumanForYou/blob/main/Data_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Cleaning and Preparation

## Overview
This section details the meticulous data cleaning and preparation process executed for the  **General Data**, **Employee Survey**, **Manager Survey**, and **time tracking datasets**. The primary objective was to enhance data integrity and ensure that the datasets were fully optimized for subsequent analysis.

The process encompassed the following key steps:

1. **Data Cleaning**: Systematically addressed missing values, rectified inconsistencies, and standardized column names to improve clarity and usability.
2. **Data Transformation**: Transposed datasets, standardized time formats, and computed essential metrics, including average time spent in the office.
3. **Data Integration**: Merged the cleaned datasets into a single, comprehensive file, facilitating a streamlined approach to analysis.

This thorough and structured process provided a robust foundation for insightful analysis, enabling a nuanced understanding of employee and managerial feedback as well as time management dynamics.


# Import necessary libraries

In [115]:
import pandas as pd
import numpy as np

# General data

In [116]:
gd = pd.read_csv('general_data.csv')
gd.head()

,Age,Attrition,BusinessTravel,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeID,Gender,...,NumCompaniesWorked,Over18,PercentSalaryHike,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,YearsAtCompany,YearsSinceLastPromotion,YearsWithCurrManager
0,51,No,Travel_Rarely,Sales,6,2,Life Sciences,1,1,Female,...,1.0,Y,11,8,0,1.0,6,1,0,0
1,31,Yes,Travel_Frequently,Research & Development,10,1,Life Sciences,1,2,Female,...,0.0,Y,23,8,1,6.0,3,5,1,4
2,32,No,Travel_Frequently,Research & Development,17,4,Other,1,3,Male,...,1.0,Y,15,8,3,5.0,2,5,0,3
3,38,No,Non-Travel,Research & Development,2,5,Life Sciences,1,4,Male,...,3.0,Y,11,8,3,13.0,5,8,7,5
4,32,No,Travel_Rarely,Research & Development,10,1,Medical,1,5,Male,...,4.0,Y,12,8,2,9.0,2,6,0,4


<h2> Check for duplicate and missing data

In [117]:
#Check for duplicate rows in the dataset
print(f"Number of duplicate rows: {gd.duplicated().sum()}")

Number of duplicate rows: 0


In [118]:
# Check for missing values in each column
gd.isnull().sum()

,0
Age,0
Attrition,0
BusinessTravel,0
Department,0
DistanceFromHome,0
Education,0
EducationField,0
EmployeeCount,0
EmployeeID,0
Gender,0


I notice that there are 19 missing entries in the **NumCompaniesWorked** column and 9 missing entries in the **TotalWorkingYears** column.

<h2> Handle missing values in 'TotalWorkingYears'

In [162]:
# Fill missing values in 'TotalWorkingYears' with values from 'YearsAtCompany' and convert to integer
gd['TotalWorkingYears'] = gd['TotalWorkingYears'].fillna(gd['YearsAtCompany']).astype(int)

In [120]:
# Confirm that there are no more missing values in 'TotalWorkingYears'
print(f"Remaining missing values in 'TotalWorkingYears': {gd['TotalWorkingYears'].isnull().sum()}")

Remaining missing values in 'TotalWorkingYears': 0


<h2>Handle missing values in 'NumCompaniesWorked'

In [121]:
missing_NC = gd[gd['NumCompaniesWorked'].isnull()][['TotalWorkingYears', 'YearsAtCompany']].assign(YearsDifference=gd['TotalWorkingYears'] - gd['YearsAtCompany']).sort_values(by='YearsDifference', ascending=False)
missing_NC

,TotalWorkingYears,YearsAtCompany,YearsDifference
3533,28,5,23
343,10,1,9
210,18,10,8
3910,10,3,7
4226,5,1,4
1312,7,3,4
647,9,7,2
799,7,5,2
2696,6,4,2
1521,6,5,1


After calculating `YearsDifference = gd['TotalWorkingYears'] - gd['YearsAtCompany']`, I identified some employees with a **difference of 0**, meaning they have only worked at this company. As a result, I will update their **NumCompaniesWorked** values to 0



In [122]:
# Update NumCompaniesWorked to 0 where TotalWorkingYears equals YearsAtCompany
gd.loc[(gd['NumCompaniesWorked'].isnull()) & (gd['TotalWorkingYears'] == gd['YearsAtCompany']),'NumCompaniesWorked'] = 0

For the remaining employees, I will update the missing **NumCompaniesWorked** values by replacing them with the mean of the existing entries in the dataset.

In [164]:
# Fill NaN values in 'NumCompaniesWorked' with the rounded mean value
gd["NumCompaniesWorked"] = gd["NumCompaniesWorked"].fillna(gd["NumCompaniesWorked"].mean().round()).astype(int)

In [124]:
# Confirm that there are no more missing values in 'NumCompaniesWorked'
print(f"Remaining missing values in 'NumCompaniesWorkeds': {gd['NumCompaniesWorked'].isnull().sum()}")

Remaining missing values in 'NumCompaniesWorkeds': 0


<h2>Descriptive statistics

In [125]:
gd.describe()

,Age,DistanceFromHome,Education,EmployeeCount,EmployeeID,JobLevel,MonthlyIncome,NumCompaniesWorked,PercentSalaryHike,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,YearsAtCompany,YearsSinceLastPromotion,YearsWithCurrManager
count,4410.000000,4410.000000,4410.000000,4410.0,4410.000000,4410.000000,4410.000000,4410.000000,4410.000000,4410.0,4410.000000,4410.000000,4410.000000,4410.000000,4410.000000,4410.000000
mean,36.923810,9.192517,2.912925,1.0,2205.500000,2.063946,65029.312925,2.692744,15.209524,8.0,0.793878,11.275737,2.799320,7.008163,2.187755,4.123129
std,9.133301,8.105026,1.023933,0.0,1273.201673,1.106689,47068.888559,2.495206,3.659108,0.0,0.851883,7.780539,1.288978,6.125135,3.221699,3.567327
min,18.000000,1.000000,1.000000,1.0,1.000000,1.000000,10090.000000,0.000000,11.000000,8.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,30.000000,2.000000,2.000000,1.0,1103.250000,1.000000,29110.000000,1.000000,12.000000,8.0,0.000000,6.000000,2.000000,3.000000,0.000000,2.000000
50%,36.000000,7.000000,3.000000,1.0,2205.500000,2.000000,49190.000000,2.000000,14.000000,8.0,1.000000,10.000000,3.000000,5.000000,1.000000,3.000000
75%,43.000000,14.000000,4.000000,1.0,3307.750000,3.000000,83800.000000,4.000000,18.000000,8.0,1.000000,15.000000,3.000000,9.000000,3.000000,7.000000
max,60.000000,29.000000,5.000000,1.0,4410.000000,5.000000,199990.000000,9.000000,25.000000,8.0,3.000000,40.000000,6.000000,40.000000,15.000000,17.000000


I found that **EmployeeCount** is consistently 1, **StandardHours** is 8, and the minimum age is 18. Consequently, the columns **Over18**, **EmployeeCount**, and **StandardHours** provide no valuable information and should be deleted.




In [126]:
gd.drop(columns=['Over18', 'EmployeeCount', 'StandardHours'], inplace=True)

<h2> Encode categorical variables

In [165]:
# Convert 'Attrition' from categorical ('Yes', 'No') to numerical (1, 0)
gd['Attrition'] = gd['Attrition'].map({'No': 0, 'Yes': 1})


In [166]:
# Convert 'Gender' from categorical ('Female', 'Male') to numerical (0, 1)
gd['Gender'] = gd['Gender'].map({'Female': 0, 'Male': 1})

In [167]:
# Convert 'BusinessTravel' to numerical values
gd['BusinessTravel'] = gd['BusinessTravel'].map({
    'Non-Travel': 0,
    'Travel_Rarely': 1,
    'Travel_Frequently': 2
})

# Manager survey

In [130]:
ms = pd.read_csv('manager_survey_data.csv')
ms.head()

,EmployeeID,JobInvolvement,PerformanceRating
0,1,3,3
1,2,2,4
2,3,3,3
3,4,2,3
4,5,3,3


<h2> Check for duplicate and missing data

In [131]:
#Check for duplicate rows in the dataset
print(f"Number of duplicate rows: {ms.duplicated().sum()}")

Number of duplicate rows: 0


In [132]:
# Check for missing values in each column
ms.isnull().sum()

,0
EmployeeID,0
JobInvolvement,0
PerformanceRating,0


# Employee survey

In [133]:
emp= pd.read_csv('employee_survey_data.csv')
emp.head()

,EmployeeID,EnvironmentSatisfaction,JobSatisfaction,WorkLifeBalance
0,1,3.0,4.0,2.0
1,2,3.0,2.0,4.0
2,3,2.0,2.0,1.0
3,4,4.0,4.0,3.0
4,5,4.0,1.0,3.0


<h2> Check for duplicate and missing data

In [134]:
#Check for duplicate rows in the dataset
print(f"Number of duplicate rows: {emp.duplicated().sum()}")

Number of duplicate rows: 0


In [135]:
# Check for missing values in each column
emp.isnull().sum()

,0
EmployeeID,0
EnvironmentSatisfaction,25
JobSatisfaction,20
WorkLifeBalance,38


<h2>Handle missing values

In [168]:
# Fill missing values in 'EnvironmentSatisfaction' with the rounded mean
emp["EnvironmentSatisfaction"] = emp["EnvironmentSatisfaction"].fillna(emp["EnvironmentSatisfaction"].mean().round())

# Fill missing values in 'JobSatisfaction' with the rounded mean
emp["JobSatisfaction"] = emp["JobSatisfaction"].fillna(emp["JobSatisfaction"].mean().round())

# Fill missing values in 'WorkLifeBalance' with the rounded mean
emp["WorkLifeBalance"] = emp["WorkLifeBalance"].fillna(emp["WorkLifeBalance"].mean().round())


# Time

## In-time Data

In [137]:
 # Load the dataset
intime = pd.read_csv('in_time.csv')
intime.head(2)

,Unnamed: 0,2015-01-01,2015-01-02,2015-01-05,2015-01-06,2015-01-07,2015-01-08,2015-01-09,2015-01-12,2015-01-13,...,2015-12-18,2015-12-21,2015-12-22,2015-12-23,2015-12-24,2015-12-25,2015-12-28,2015-12-29,2015-12-30,2015-12-31
0,1,NaN,2015-01-02 09:43:45,2015-01-05 10:08:48,2015-01-06 09:54:26,2015-01-07 09:34:31,2015-01-08 09:51:09,2015-01-09 10:09:25,2015-01-12 09:42:53,2015-01-13 10:13:06,...,NaN,2015-12-21 09:55:29,2015-12-22 10:04:06,2015-12-23 10:14:27,2015-12-24 10:11:35,NaN,2015-12-28 10:13:41,2015-12-29 10:03:36,2015-12-30 09:54:12,2015-12-31 10:12:44
1,2,NaN,2015-01-02 10:15:44,2015-01-05 10:21:05,NaN,2015-01-07 09:45:17,2015-01-08 10:09:04,2015-01-09 09:43:26,2015-01-12 10:00:07,2015-01-13 10:43:29,...,2015-12-18 10:37:17,2015-12-21 09:49:02,2015-12-22 10:33:51,2015-12-23 10:12:10,NaN,NaN,2015-12-28 09:31:45,2015-12-29 09:55:49,2015-12-30 10:32:25,2015-12-31 09:27:20


<h2> Check for duplicate and missing data

In [138]:
#Check for duplicate rows in the dataset
print(f"Number of duplicate rows: {intime.duplicated().sum()}")

Number of duplicate rows: 0


In [139]:
#Checking for Missing Values
intime.isnull().sum()

,0
Unnamed: 0,0
2015-01-01,4410
2015-01-02,209
2015-01-05,206
2015-01-06,228
...,...
2015-12-25,4410
2015-12-28,234
2015-12-29,230
2015-12-30,265


<h2>Handle Missing Values

In [140]:
# Transpose the DataFrame to switch rows with columns
intime_T = intime.T

In [141]:
# Filling Missing Values with Mode
for employee in intime_T:
    value_to_fill = intime_T[employee].value_counts().sort_values(ascending=False).index[1]
    intime_T[employee] = intime_T[employee].fillna(value_to_fill)

In [142]:
# Converting Time to Minutes in a Day
for employee in intime_T:
    datetime_data = pd.to_datetime(intime_T[employee])  # Convert to datetime format
    hour_data = datetime_data.dt.strftime('%H').apply(lambda x: int(x))  # Extract hours as integers
    min_data = datetime_data.dt.strftime('%M').apply(lambda x: int(x))  # Extract minutes as integers
    data_to_return = hour_data * 60 + min_data  # Convert to minutes since the start of the day
    intime_T[employee] = data_to_return  # Update the DataFrame with the new values

## Out-time Data

In [143]:
 # Load the dataset
outime = pd.read_csv('out_time.csv')
outime.head(2)

,Unnamed: 0,2015-01-01,2015-01-02,2015-01-05,2015-01-06,2015-01-07,2015-01-08,2015-01-09,2015-01-12,2015-01-13,...,2015-12-18,2015-12-21,2015-12-22,2015-12-23,2015-12-24,2015-12-25,2015-12-28,2015-12-29,2015-12-30,2015-12-31
0,1,NaN,2015-01-02 16:56:15,2015-01-05 17:20:11,2015-01-06 17:19:05,2015-01-07 16:34:55,2015-01-08 17:08:32,2015-01-09 17:38:29,2015-01-12 16:58:39,2015-01-13 18:02:58,...,NaN,2015-12-21 17:15:50,2015-12-22 17:27:51,2015-12-23 16:44:44,2015-12-24 17:47:22,NaN,2015-12-28 18:00:07,2015-12-29 17:22:30,2015-12-30 17:40:56,2015-12-31 17:17:33
1,2,NaN,2015-01-02 18:22:17,2015-01-05 17:48:22,NaN,2015-01-07 17:09:06,2015-01-08 17:34:04,2015-01-09 16:52:29,2015-01-12 17:36:48,2015-01-13 18:00:13,...,2015-12-18 18:31:28,2015-12-21 17:34:16,2015-12-22 18:16:35,2015-12-23 17:38:18,NaN,NaN,2015-12-28 17:08:38,2015-12-29 17:54:46,2015-12-30 18:31:35,2015-12-31 17:40:58


<h2> Check for duplicate and missing data

In [144]:
#Check for duplicate rows in the dataset
print(f"Number of duplicate rows: {outime.duplicated().sum()}")

Number of duplicate rows: 0


In [145]:
#Checking for Missing Values
intime.isnull().sum()

,0
Unnamed: 0,0
2015-01-01,4410
2015-01-02,209
2015-01-05,206
2015-01-06,228
...,...
2015-12-25,4410
2015-12-28,234
2015-12-29,230
2015-12-30,265


<h2>Handle Missing Values

In [146]:
outime_T = outime.T  # Transpose the 'outime' DataFrame
for employee in outime_T:
    value_to_fill = outime_T[employee].value_counts().sort_values(ascending=False).index[1]
    outime_T[employee] = outime_T[employee].fillna(value_to_fill)  # Fill missing values

In [147]:
# Converting Out-time to Minutes
for employee in outime_T:
    datetime_data = pd.to_datetime(outime_T[employee])  # Convert to datetime
    hour_data = datetime_data.dt.strftime('%H').apply(lambda x: int(x))  # Get the hours
    min_data = datetime_data.dt.strftime('%M').apply(lambda x: int(x))  # Get the minutes
    data_to_return = hour_data * 60 + min_data  # Convert to total minutes
    outime_T[employee] = data_to_return  # Update the DataFrame

## Time Spent in the Office


In [154]:
# Calculate the time spent in the Office
office_time_spent = outime_T - intime_T
office_time_spent = office_time_spent.iloc[1:, :]  # Remove the 'Unnamed: 0' row

In [155]:
# Calculate the mean time spent for each employee
agg_time_spent = pd.DataFrame(office_time_spent.T.mean(axis=1), columns=['Time_spent'])

In [157]:
# Combine gd and agg_time_spent DataFrames column-wise into new df
df= pd.concat([gd,agg_time_spent],axis=1)

# Final Version

In [158]:
# Merge df with emp and mg DataFrames on 'EmployeeID' to create a combined dataset
data = df.merge(emp, on='EmployeeID').merge(ms, on='EmployeeID')

In [160]:
#checking
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4410 entries, 0 to 4409
Data columns (total 27 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Age                      4410 non-null   int64  
 1   Attrition                4410 non-null   int64  
 2   BusinessTravel           4410 non-null   int64  
 3   Department               4410 non-null   object 
 4   DistanceFromHome         4410 non-null   int64  
 5   Education                4410 non-null   int64  
 6   EducationField           4410 non-null   object 
 7   EmployeeID               4410 non-null   int64  
 8   Gender                   4410 non-null   int64  
 9   JobLevel                 4410 non-null   int64  
 10  JobRole                  4410 non-null   object 
 11  MaritalStatus            4410 non-null   object 
 12  MonthlyIncome            4410 non-null   int64  
 13  NumCompaniesWorked       4410 non-null   int64  
 14  PercentSalaryHike       

In [161]:
#Display
data.head()

,Age,Attrition,BusinessTravel,Department,DistanceFromHome,Education,EducationField,EmployeeID,Gender,JobLevel,...,TrainingTimesLastYear,YearsAtCompany,YearsSinceLastPromotion,YearsWithCurrManager,Time_spent,EnvironmentSatisfaction,JobSatisfaction,WorkLifeBalance,JobInvolvement,PerformanceRating
0,51,0,1,Sales,6,2,Life Sciences,1,0,1,...,6,1,0,0,440.287356,3.0,4.0,2.0,3,3
1,31,1,2,Research & Development,10,1,Life Sciences,2,0,1,...,3,5,1,4,461.620690,3.0,2.0,4.0,2,4
2,32,0,2,Research & Development,17,4,Other,3,1,4,...,2,5,0,3,419.659004,2.0,2.0,1.0,3,3
3,38,0,0,Research & Development,2,5,Life Sciences,4,1,3,...,5,8,7,5,428.659004,4.0,4.0,3.0,2,3
4,32,0,1,Research & Development,10,1,Medical,5,1,1,...,2,6,0,4,479.984674,4.0,1.0,3.0,3,3


In [159]:
# Save the cleaned data to a CSV file named 'data.csv' without including the index
data.to_csv('data.csv', index=False)